In [1]:
import pandas as pd

In [2]:
import numpy as np

In [3]:
import matplotlib.pyplot as plt

In [4]:
import seaborn as sns

In [5]:
import joblib

In [6]:
import time

In [7]:
import streamlit as st

In [8]:
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV

In [9]:
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, learning_curve

In [10]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [11]:
from sklearn.linear_model import LogisticRegression

In [12]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier

In [13]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc

In [14]:
from sklearn.utils.class_weight import compute_class_weight

#Loading our dataset

In [16]:
df = pd.read_csv("C:/Users/arist/Desktop/survey lung cancer.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/arist/OneDrive/Desktop/Data Mining/survey lung cancer.csv'

In [ ]:
df.head(20)

#EXPLORE THE DATA

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.isnull()

In [ ]:
df.tail()

In [ ]:
df.dtypes

#Encode categorical variables

In [ ]:
label_encoder = LabelEncoder()

In [ ]:
df['LUNG_CANCER'] = label_encoder.fit_transform(df['LUNG_CANCER'])

In [ ]:
df['GENDER'] = label_encoder.fit_transform(df['GENDER'])  

In [ ]:
df.head()

# Step 4: Feature Engineering - Creating new features
# Example: Age squared to capture non-linear patterns

In [ ]:
df['AGE_SQUARED'] = df['AGE'] ** 2

# Step 6: Handle class imbalance using SMOTE

 #Split the data into features and target

In [ ]:
from imblearn.over_sampling import SMOTE

In [ ]:
from collections import Counter

In [ ]:
x = df.drop(columns=['LUNG_CANCER'])

In [ ]:
y = df['LUNG_CANCER'] 

In [ ]:
smote = SMOTE(random_state=42)   #imbalance.

In [ ]:
x_resampled, y_resampled = smote.fit_resample(x, y)

#Handle class imbalance

In [ ]:
class_weights = compute_class_weight('balanced', classes=np.unique(y), y=y)
class_weights_dict = {i: class_weights[i] for i in range(len(class_weights))}
print("Class Weights:", class_weights_dict)

#Split into training and testing sets

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

#Standardize features

In [ ]:
from sklearn.preprocessing import StandardScaler 

In [ ]:
scaler = StandardScaler() 

In [ ]:
df_scaled = pd.DataFrame(scaler.fit_transform(df.select_dtypes(include=['number'])), columns=df.select_dtypes(include=['number']).columns) 

In [ ]:
print(df_scaled.head()) 

In [ ]:
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x_scaled, y, test_size=0.2, random_state=42)

In [ ]:
logistic_model = LogisticRegression()

In [ ]:
start_time = time.time()

In [ ]:
end_time = time.time()
print(f"Logistic Regression Training Time: {end_time - start_time:.2f} seconds")

In [ ]:
logistic_model.fit(x_train, y_train)

In [ ]:
y_pred = logistic_model.predict(x_test)

In [ ]:
print(y_pred)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

In [ ]:
print(f"Model Accuracy: {accuracy:.2f}")

In [ ]:
# Train Additional Models

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

In [ ]:
gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)

In [ ]:
rf_model.fit(x_train, y_train)
gb_model.fit(x_train, y_train)

In [ ]:
# Create Ensemble Model

In [ ]:
ensemble_model = VotingClassifier(estimators=[('lr', logistic_model), ('rf', rf_model), ('gb', gb_model)], voting='soft')

In [ ]:
ensemble_model.fit(x_train, y_train)

In [ ]:
# Save the trained models

In [ ]:
joblib.dump(logistic_model, 'logistic_regression_model.pkl')

In [ ]:
joblib.dump(ensemble_model, 'ensemble_model.pkl')

In [ ]:
# Make predictions

In [ ]:
y_pred = ensemble_model.predict(x_test)

In [ ]:
print(y_pred)

In [ ]:
# Evaluate the model

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print("Ensemble Model Accuracy:", accuracy)
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
conf_matrix = confusion_matrix(y_test, y_pred)
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=['No Cancer', 'Cancer'], yticklabels=['No Cancer', 'Cancer'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# ROC Curve

In [ ]:
y_prob = ensemble_model.predict_proba(x_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', label=f'ROC Curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

In [ ]:
# Feature Importance Analysis

In [ ]:
feature_importance = rf_model.feature_importances_
feature_names = x.columns
feature_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importance})
feature_df = feature_df.sort_values(by='Importance', ascending=False)


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_df, palette='viridis')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Feature Importance in Random Forest')
plt.show()

In [ ]:
# Hyperparameter Tuning with Parallel Processing

In [ ]:
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'lbfgs']
}
grid_search = GridSearchCV(LogisticRegression(class_weight='balanced', max_iter=1000), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(x_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best Cross-Validation Score:", grid_search.best_score_)

In [ ]:
# Cross-Validation

In [ ]:
cv_scores = cross_val_score(ensemble_model, x, y, cv=5, scoring='accuracy', n_jobs=-1)
print("Cross-Validation Accuracy Scores:", cv_scores)
print("Mean Cross-Validation Accuracy:", np.mean(cv_scores))


In [ ]:
# Learning Curve

In [ ]:
def plot_learning_curve(estimator, x, y):
    train_sizes, train_scores, test_scores = learning_curve(estimator, x, y, cv=5, scoring='accuracy', n_jobs=-1, train_sizes=np.linspace(0.1, 1.0, 10))
    train_mean = np.mean(train_scores, axis=1)
    test_mean = np.mean(test_scores, axis=1)
    
    plt.figure(figsize=(8, 6))
    plt.plot(train_sizes, train_mean, 'o-', label='Training Score')
    plt.plot(train_sizes, test_mean, 'o-', label='Validation Score')
    plt.xlabel('Training Size')
    plt.ylabel('Accuracy')
    plt.title('Learning Curve')
    plt.legend()
    plt.show()

plot_learning_curve(logistic_model, x_train, y_train)

In [ ]:
# Final Model Training

In [ ]:
final_model = LogisticRegression(C=grid_search.best_params_['C'], solver=grid_search.best_params_['solver'], class_weight='balanced', max_iter=1000)
final_model.fit(x_train, y_train)

In [ ]:
print("Final Model Training Completed!")

In [ ]:
joblib.dump(logistic_model, 'logistic_regression_model.pkl')

In [ ]:
joblib.dump(ensemble_model, 'ensemble_model.pkl')